In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain.chat_models import init_chat_model
from typing import TypedDict

from typing import Annotated
from langgraph.graph.message import add_messages

In [16]:
llm = init_chat_model("openai:gpt-5-mini")


class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state:State) -> State:
    return {"messages": [llm.invoke(state['messages'])]}



In [17]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()


In [18]:
message = {"role": "user", "content": "Who was the first person to walk on moon? Print only the name"}
response = graph.invoke({"messages": [message]})
response['messages']

[HumanMessage(content='Who was the first person to walk on moon? Print only the name', additional_kwargs={}, response_metadata={}, id='3c574761-775c-40d2-adf7-d7bec86a01d8'),
 AIMessage(content='Neil Armstrong', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 20, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUcCLOtpbximIFnnSCTtZgQd1luxB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d8d1d-f132-7cf1-a871-5f211fd3f3de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 75, 'total_tokens': 95, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output

In [19]:
state = None
while True:
    in_message = input('You: ')
    if in_message.lower() in ["quit", "exit"]:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})
    
    state = graph.invoke(state)
    print("Bot: ", state["messages"][-1].content)

Bot:  Neil Armstrong
Bot:  1969
Bot:  NASA
Bot:  Buzz Aldrin (on the Moon) and Michael Collins (in the Command Module).
